# Digital Farm Management Portal: Poultry Biosecurity Disease Outbreak Risk Prediction
### Act as a Machine Learning Engineer

This Google Colab notebook implements an end-to-end Machine Learning pipeline to predict the disease outbreak risk level in poultry farms based on environmental and farm management factors. It is fully enhanced with cost-sensitive class balancing to handle real-world minority disease outbreak events.

#### Features in the Dataset:
1. **Temperature (°C)**: Inside temperature of the poultry house.
2. **Humidity (%)**: Relative humidity levels.
3. **Mortality_Rate (%)**: Daily mortality rate of the poultry flock.
4. **Egg_Production**: Standard daily count of egg production.
5. **Amount_of_Feeding (kg/day)**: Feed intake of the flock.
6. **Active_Birds**: Total live poultry population in the house.
7. **Bird_Age (days)**: Flock age in days.
8. **Vaccination_Status**: Flock vaccination category ('Vaccinated' or 'Unvaccinated').
9. **Risk_Level**: Target variable to predict ('Low', 'Medium', or 'High' risk).

---


## 1. Import Required Libraries & Set Styles
First, we will import all essential Python packages for data analysis, visualization, modeling, and serialization.


In [ ]:
import os
import csv
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Visual settings
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('Libraries imported and styles configured successfully!')


## 2. Scientifically Grounded Synthetic Dataset Generation
To enable this notebook to run completely self-contained in Google Colab without requiring any manual file uploads, we generate a highly realistic, biologically and environmentally informed dataset of **5,000 farm records**.

**Core Scientific Rules Modeled:**
- **Heat Stress**: High temperature (>30°C) and high humidity (>80%) combined limit poultry's heat dissipation, raising stress levels.
- **Feed and Egg drop**: Disease causes inappetence (feed intake drops 25% - 50%) and standard egg production drops (20% - 80%).
- **Vaccination**: Unvaccinated birds have a significant risk penalty under environmental stress.
- **Mortality Rate**: Directly indicates ongoing disease outbreaks (spikes to >0.5% daily).


In [ ]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

num_records = 5000
headers = [
    'Temperature (°C)', 'Humidity (%)', 'Mortality_Rate (%)', 'Egg_Production',
    'Amount_of_Feeding (kg/day)', 'Active_Birds', 'Bird_Age (days)', 'Vaccination_Status', 'Risk_Level'
]

data = []

for i in range(num_records):
    active_birds = random.randint(1000, 15000)
    bird_age = random.randint(1, 90)
    temperature = round(random.normalvariate(22.0, 5.0), 1)
    temperature = max(10.0, min(40.0, temperature))
    
    humidity = round(random.normalvariate(60.0, 12.0), 1)
    humidity = max(20.0, min(95.0, humidity))
    
    vaccination_status = 'Vaccinated' if random.random() < 0.80 else 'Unvaccinated'
    
    # Base health logic
    health_score = 0.0
    if temperature > 32.0 or temperature < 14.0:
        health_score += 0.25
        if humidity > 80.0 or humidity < 40.0:
            health_score += 0.2
    elif humidity > 85.0 or humidity < 35.0:
        health_score += 0.15
        
    if vaccination_status == 'Unvaccinated' and health_score > 0:
        health_score += 0.3
        
    rand_val = random.random()
    if rand_val < 0.05:
        health_score += 0.8  # severe shock
    elif rand_val < 0.15:
        health_score += 0.35 # moderate shock
        
    if health_score >= 0.7:
        health_status = 'Severe Outbreak'
    elif health_score >= 0.3:
        health_status = 'Mild Stress'
    else:
        health_status = 'Healthy'
        
    # Mortality
    if health_status == 'Healthy':
        mortality_rate = round(random.uniform(0.005, 0.05), 3)
    elif health_status == 'Mild Stress':
        mortality_rate = round(random.uniform(0.06, 0.4), 3)
    else:
        mortality_rate = round(random.uniform(0.5, 4.5), 3)
        
    # Feed intake
    if bird_age < 15:
        base_feed_per_bird = random.uniform(0.03, 0.05)
    elif bird_age < 45:
        base_feed_per_bird = random.uniform(0.08, 0.11)
    else:
        base_feed_per_bird = random.uniform(0.12, 0.14)
        
    expected_feeding = active_birds * base_feed_per_bird
    if health_status == 'Healthy':
        feed_ratio = random.uniform(0.95, 1.03)
    elif health_status == 'Mild Stress':
        feed_ratio = random.uniform(0.80, 0.92)
    else:
        feed_ratio = random.uniform(0.45, 0.75)
    amount_of_feeding = round(expected_feeding * feed_ratio, 1)
    
    # Egg Production
    if bird_age <= 20:
        egg_production = 0
    else:
        base_lay_rate = random.uniform(0.70, 0.85)
        expected_eggs = active_birds * base_lay_rate
        if health_status == 'Healthy':
            egg_ratio = random.uniform(0.95, 1.02)
        elif health_status == 'Mild Stress':
            egg_ratio = random.uniform(0.70, 0.90)
        else:
            egg_ratio = random.uniform(0.15, 0.55)
        egg_production = int(expected_eggs * egg_ratio)
        
    # Logical risk target assignment
    actual_feed_ratio = amount_of_feeding / expected_feeding
    if bird_age > 20:
        actual_egg_ratio = egg_production / (active_birds * 0.75)
    else:
        actual_egg_ratio = 1.0
        
    if mortality_rate >= 0.5 or (mortality_rate >= 0.3 and actual_feed_ratio < 0.75) or (actual_feed_ratio < 0.65):
        risk_level = 'High'
    elif mortality_rate >= 0.1 or actual_feed_ratio < 0.90 or actual_egg_ratio < 0.80 or (temperature > 34.0 and vaccination_status == 'Unvaccinated') or (humidity > 85.0 and temperature > 30.0 and vaccination_status == 'Unvaccinated'):
        risk_level = 'Medium'
    else:
        risk_level = 'Low'
        
    # 2% Noise
    if random.random() < 0.02:
        risk_level = random.choice(['Low', 'Medium', 'High'])
        
    data.append([
        temperature, humidity, mortality_rate, egg_production, amount_of_feeding,
        active_birds, bird_age, vaccination_status, risk_level
    ])

# Create Pandas DataFrame
df = pd.DataFrame(data, columns=headers)
df.to_csv('poultry_biosecurity_dataset.csv', index=False)
print(f'Successfully generated {len(df)} poultry biosecurity farm records!')
print('Class Counts:\n', df['Risk_Level'].value_counts())
df.head()


## 3. Exploratory Data Analysis (EDA)
We will check for missing values and plot distributions and correlation heatmaps to understand our features. Note the classic class imbalance: Low risk is the dominant state, while Medium and High risk represents rare biosecurity incidents.


In [ ]:
# 3.1 Missing Value Analysis
print('--- Missing Values Analysis ---')
missing_report = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print(missing_report)

# 3.2 Distribution Plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Feature Distributions & Classes', fontsize=18, fontweight='bold')

sns.histplot(data=df, x='Temperature (°C)', hue='Risk_Level', multiple='stack', palette='crest', ax=axes[0,0], kde=True)
axes[0,0].set_title('Temperature Distribution by Risk', fontsize=13, fontweight='semibold')

sns.histplot(data=df, x='Humidity (%)', hue='Risk_Level', multiple='stack', palette='mako', ax=axes[0,1], kde=True)
axes[0,1].set_title('Humidity Distribution by Risk', fontsize=13, fontweight='semibold')

sns.histplot(data=df, x='Mortality_Rate (%)', hue='Risk_Level', multiple='stack', palette='rocket_r', ax=axes[1,0], log_scale=(False, True))
axes[1,0].set_title('Mortality Rate (Log-Scale count)', fontsize=13, fontweight='semibold')

sns.countplot(data=df, x='Risk_Level', order=['Low', 'Medium', 'High'], palette=['#10b981', '#f59e0b', '#ef4444'], ax=axes[1,1])
axes[1,1].set_title('Target Outbreak Risk Classes Count (Visual Imbalance)', fontsize=13, fontweight='semibold')

plt.tight_layout()
plt.show()

# 3.3 Correlation Heatmap
df_corr = df.copy()
df_corr['Vaccination_Status_Encoded'] = df_corr['Vaccination_Status'].map({'Vaccinated': 1, 'Unvaccinated': 0})
df_corr['Risk_Level_Encoded'] = df_corr['Risk_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
df_corr_numeric = df_corr.drop(columns=['Vaccination_Status', 'Risk_Level'])

plt.figure(figsize=(12, 10))
sns.heatmap(df_corr_numeric.corr(), annot=True, fmt='.2f', cmap=sns.diverging_palette(230, 20, as_cmap=True), linewidths=0.5, square=True)
plt.title('Correlation Matrix of Poultry Farm Features', fontsize=16, fontweight='bold', pad=15)
plt.show()


## 4. Data Preprocessing
We map the target labels (`Risk_Level` -> integers), binary-encode `Vaccination_Status`, split the data (80% train, 20% test), and scale numeric features using `StandardScaler`.


In [ ]:
# Map target risk levels
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_risk_map = {0: 'Low', 1: 'Medium', 2: 'High'}
df['Risk_Level_Encoded'] = df['Risk_Level'].map(risk_map)

# Map Vaccination status
df['Vaccination_Status_Encoded'] = df['Vaccination_Status'].map({'Vaccinated': 1, 'Unvaccinated': 0})

feature_cols = [
    'Temperature (°C)', 'Humidity (%)', 'Mortality_Rate (%)', 'Egg_Production',
    'Amount_of_Feeding (kg/day)', 'Active_Birds', 'Bird_Age (days)', 'Vaccination_Status_Encoded'
]

X = df[feature_cols]
y = df['Risk_Level_Encoded']

# Train-test split (stratifying by target to preserve class ratios in splits)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Scale numeric variables
numeric_cols = [col for col in feature_cols if col != 'Vaccination_Status_Encoded']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])
print('Data split and scaling complete!')


## 5. Model Training & Cost-Sensitive Imbalance Comparison
We address the class imbalance directly in model architectures using cost-sensitive learning weights:
1. **Logistic Regression** (using `class_weight='balanced'` to scale loss weights)
2. **Decision Tree** (using `class_weight='balanced'`)
3. **Random Forest** (using `class_weight='balanced'`)
4. **XGBoost** (using computed balanced sample weights during training)

To evaluate them accurately, we track **Balanced Accuracy** (average recall per class) and **Macro F1-Score** (weights minority classes equally with Low risk) rather than relying solely on default standard Accuracy.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
}

comparison_data = []
confusion_matrices = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

# Calculate class weights for XGBoost training
xgb_weights = compute_sample_weight(class_weight='balanced', y=y_train)

for idx, (name, model) in enumerate(models.items()):
    # Train
    if name == 'XGBoost':
        model.fit(X_train_scaled, y_train, sample_weight=xgb_weights)
    else:
        model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate metrics
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    precision, recall, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')
    
    comparison_data.append({
        'Model': name,
        'Accuracy': acc,
        'Balanced Accuracy': bal_acc,
        'Precision (W)': precision,
        'Recall (W)': recall,
        'F1 Score (Weighted)': f1_w,
        'F1 Score (Macro)': f1_m
    })
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'], ax=axes[idx], cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
    axes[idx].set_title(f'Confusion Matrix: {name}', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.suptitle('Model Evaluation: Confusion Matrices', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Compare models in a beautiful table
comparison_df = pd.DataFrame(comparison_data)
print('\n================================== MODEL COMPARISON METRICS ==================================')
print(comparison_df.to_string(index=False, formatters={
    'Accuracy': '{:.4f}'.format,
    'Balanced Accuracy': '{:.4f}'.format,
    'Precision (W)': '{:.4f}'.format,
    'Recall (W)': '{:.4f}'.format,
    'F1 Score (Weighted)': '{:.4f}'.format,
    'F1 Score (Macro)': '{:.4f}'.format
}))
print('==============================================================================================')

best_idx = comparison_df['F1 Score (Macro)'].idxmax()
best_model_name = comparison_df.iloc[best_idx]['Model']
best_model_obj = models[best_model_name]
print(f'\nBest-Performing Model (by Macro F1): {best_model_name} with Macro F1-Score of {comparison_df.iloc[best_idx]["F1 Score (Macro"]:.4f}!')


## 6. Feature Importance Rankings
Using our best-performing model, we plot and analyze the feature importances to see which factors contribute most to predicting biosecurity outbreaks.


In [ ]:
# Feature Importance
importance_model_name = best_model_name
importance_model = best_model_obj

if importance_model_name not in ['Random Forest', 'XGBoost', 'Decision Tree']:
    importance_model_name = 'Random Forest'
    importance_model = models['Random Forest']

importances = importance_model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [feature_cols[i] for i in indices]
sorted_importances = importances[indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_importances, y=sorted_features, palette='viridis')
plt.title(f'Feature Importance Ranking ({importance_model_name})', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Relative Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

print('--- Feature Importance Table ---')
for i in range(len(sorted_features)):
    print(f'{i+1}. {sorted_features[i]:<30} : {sorted_importances[i]:.4f}')


## 7. How Temperature and Humidity Biological Factors Contribute to Outbreaks

### **1. Temperature's Role:**
- **Heat Stress**: Poultry do not have sweat glands. When temperature rises above **30°C**, they experience heat stress, panting excessively. This dries out the respiratory passages, reducing their natural mechanical immunity against inhaled viral particles (e.g. Bird Flu / Avian Influenza, Newcastle Disease).
- **Cold Stress**: Temperatures below **14°C** trigger cold stress. Huddled together to conserve heat, birds are in closer physical contact, facilitating extremely rapid transmission of standard pathogens across the flock.

### **2. Humidity's Role:**
- **High Humidity (>80%)**: Combined with high temperature, it restricts the birds' capacity for evaporative cooling via panting, compounding heat stress and triggering immediate respiratory arrest.
- **High Moisture in Litter**: Wet conditions promote the rapid multiplication of severe parasites (like Coccidiosis) and bacteria (like Salmonella), while generating toxic Ammonia levels which irritate bird lungs.
- **Low Humidity (<40%)**: Causes dusty, highly dry conditions that irritate the tracheal membranes of active birds, making them highly susceptible to secondary bacterial respiratory diseases.

### **3. Multiplicative Interaction:**
- Environmental stress is not additive, but **multiplicative** (e.g., Temperature $\times$ Humidity interactions). This non-linear relationship is why tree-based models like **Random Forest** and **XGBoost** show exceptional predictive power compared to standard linear classification approaches.


## 8. Save Trained Model and Preprocessor via Pickle
We save the best-performing model, numerical scaler, features, and target mapping into a single `.pkl` dictionary.


In [ ]:
output_model_path = 'poultry_risk_model.pkl'
pickle_payload = {
    'model_name': best_model_name,
    'model': best_model_obj,
    'scaler': scaler,
    'numeric_cols': numeric_cols,
    'feature_cols': feature_cols,
    'risk_map': risk_map,
    'inv_risk_map': inv_risk_map
}

with open(output_model_path, 'wb') as f:
    pickle.dump(pickle_payload, f)
print(f'Successfully saved model dictionary package to {output_model_path}!')


## 9. Deployment: Custom Prediction Function
Now we create the required `predict_risk` function that returns both the **Risk Score** (probability of elevated disease risk) and the **Risk Level** ('Low', 'Medium', 'High').


In [ ]:
def predict_risk(
    temperature,
    humidity,
    mortality_rate,
    egg_production,
    amount_of_feeding,
    active_birds,
    bird_age=35,
    vaccination_status='Vaccinated'
):
    """
    Predicts the disease outbreak risk level in poultry farms.
    
    Returns:
    --------
    risk_score : float (0.0 to 1.0)
    risk_level : str ('Low', 'Medium', 'High')
    """
    # Load pickled model payload
    with open('poultry_risk_model.pkl', 'rb') as f:
        payload = pickle.load(f)
        
    model = payload['model']
    scaler = payload['scaler']
    numeric_cols = payload['numeric_cols']
    feature_cols = payload['feature_cols']
    inv_risk_map = payload['inv_risk_map']
    
    # Preprocess inputs
    vacc_encoded = 1 if vaccination_status.lower() == 'vaccinated' else 0
    
    # Build input DataFrame
    input_data = pd.DataFrame([{
        'Temperature (°C)': float(temperature),
        'Humidity (%)': float(humidity),
        'Mortality_Rate (%)': float(mortality_rate),
        'Egg_Production': int(egg_production),
        'Amount_of_Feeding (kg/day)': float(amount_of_feeding),
        'Active_Birds': int(active_birds),
        'Bird_Age (days)': int(bird_age),
        'Vaccination_Status_Encoded': int(vacc_encoded)
    }])
    
    # Reorder columns
    input_data = input_data[feature_cols]
    
    # Scale numerical values
    input_data_scaled = input_data.copy()
    input_data_scaled[numeric_cols] = scaler.transform(input_data[numeric_cols])
    
    # Predict probabilities
    probabilities = model.predict_proba(input_data_scaled)[0]
    pred_idx = model.predict(input_data_scaled)[0]
    
    # Risk Score = probability of elevated risk (1.0 - P(Low))
    risk_score = round(1.0 - probabilities[0], 4)
    risk_level = inv_risk_map[pred_idx]
    
    return float(risk_score), str(risk_level)

print('predict_risk function successfully defined!')


## 10. Live Interactive Prediction Playground
Test the `predict_risk` function under different simulated scenarios below.


In [ ]:
print('--- Test Scenario 1: Optimal Healthy Farm ---')
score, level = predict_risk(temperature=21.5, humidity=55.0, mortality_rate=0.015, egg_production=8200, amount_of_feeding=1250, active_birds=10000)
print(f'Risk Score: {score:.4f} | Risk Level: {level}')
print('-' * 60)

print('--- Test Scenario 2: Heat Stressed Flocks with Feed Drop ---')
score, level = predict_risk(temperature=32.5, humidity=75.0, mortality_rate=0.18, egg_production=7000, amount_of_feeding=1050, active_birds=10000, vaccination_status='Unvaccinated')
print(f'Risk Score: {score:.4f} | Risk Level: {level}')
print('-' * 60)

print('--- Test Scenario 3: Severe Disease Outbreak Spike ---')
score, level = predict_risk(temperature=36.0, humidity=82.0, mortality_rate=1.85, egg_production=2500, amount_of_feeding=620, active_birds=10000, vaccination_status='Unvaccinated')
print(f'Risk Score: {score:.4f} | Risk Level: {level}')
